<h1> Task 3

Part 1

In [45]:
# scaled dot-product attention
import numpy as np

'''
Compute softmax along last axis of input array

Parameters:
x: np.ndarray
    Input array of arbitrary shape

Returns:
np.ndarray
    Softmax probabilities of same shape as x
'''
def softmax(x):
  exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
  return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

'''
Compute scaled dot-product attention

Parameters:
  Q: Queries matrix of shape (seq_len_q, d_k)
  K: Keys matrix of shape (seq_len_k, d_k)
  V: Values matrix of shape (seq_len_k, d_v)
Returns:
  output: Attention output of shape (seq_len_q, d_v)
  attention_weights: Weights assigned to each value (seq_len_q, seq_len_k)
'''

def scaled_dot_product_attention(Q, K, V):
  # step 1 - compute raw attention scores (similarity of Q to K)
  scores = np.dot(Q, K.T) # (seq_len_q, seq_len_k)

  # step 2 - scale by sqrt(d_k) - prevents large values
  d_k = K.shape[-1]
  scaled_scores = scores / np.sqrt(d_k)

  # step 3 - apply softmax to get attention weights
  attention_weights = softmax(scaled_scores)

  # step 4 - multiply attention weights with V to get final output
  output = np.dot(attention_weights, V)

  return output, attention_weights

# test matricies
Q = np.array([[1, 0, 1],
            [0, 1, 0]]) # 2 queries of dimenesion 3

K = np.array([[1, 0, 1],
            [0, 1, 0],
            [1, 1, 0]]) # 3 keys of dimention 3

V = np.array([[1, 2],
            [3, 4],
            [5, 6]]) # 3 values of dimension 2

# compute attention
output, weights = scaled_dot_product_attention(Q, K, V)

print("Output:\n", output)
print("Attention Weights:\n", weights)

Output:
 [[2.53252575 3.53252575]
 [3.34248367 4.34248367]]
Attention Weights:
 [[0.53289684 0.16794345 0.29915971]
 [0.21917211 0.39041395 0.39041395]]


Scaled dot-product attention computes attention scores by taking the dot product of query and key vectors, scaling by the square root of the key dimension, and applying a softmax to obtain attention weights. These weights are then used to compute a weighted sum of value vectors, allowing the model to focus on relevant parts of the input sequence. The tested output shows how much each query focuses on each key. Query 1 most attends to key 1 and key 3 and query 2 mostly attends to key 2 and key 3. The output vectors are weighted sums of the value based on these attention weights.

Part 2

Encoder selected:
Seq2Seq + Attention

The encoder in a Seq2Seq model with attention is an LSTM network that processes the input sequence token by token and produces two outputs: the hidden states at each time step and the final hidden and cell states. The hidden states capture contextual information from each input token, while the final states summarize the full sequence, serving as the initial state for the decoder.

The decoder is also an LSTM network that generates the output sequence one token at a time. At each decoding step, it uses the previous token (or the previous prediction during inference) and its current hidden state to predict the next token.

Attention is added to the decoder to allow it to focus on relevant parts of the input sequence at each decoding step. Instead of relying just on the encoder’s final hidden state, attention calculates a set of weights (alignment scores) over all encoder hidden states. These weights determine how much influence each encoder hidden state should have when predicting the next output token. The context vector is computed as a weighted sum of encoder hidden states and is fed into the decoder along with its current hidden state. This mechanism helps the decoder capture long-range dependencies, handle variable-length inputs more effectively, and improve translation or sequence generation quality.

In [8]:
# encoder-decoder seq2seq
import tensorflow as tf
from tensorflow.keras import layers

# attention layer
class Attention(layers.Layer):
  '''
  Parameters:
    query: decoder hidden state (batch_size, seq_len_dec, units)
    values: encoder outputs (batch_size, seq_len_enc, units)

  Returns:
    context: weighted sum of encoder outputs
    weights: attention weights
  '''
  def call(self, query, values):
    # step 1 - compute scores (similarity of query to keys)
    scores = tf.matmul(query, values, transpose_b=True)

    # step 2 - scale scores by sqrt(d_k) (Luong attention)
    d_k = tf.cast(tf.shape(values)[-1], tf.float32)
    scaled_scores = scores / tf.math.sqrt(d_k)

    # step 3 - softmax to get attention weights
    weights = tf.nn.softmax(scaled_scores, axis=-1) # sum across encoder sequence

    # step 4 - compute context vector as weighted sum of encoder outputs
    context = tf.matmul(weights, values) # (batch, seq_len_dec, units)
    return context, weights

In [47]:
# encoder
'''
Builds an LSTM encoder

Parameters:
  vocab_size: size of input vocabulary
  embedding_dim: size of embedding vectors
  units: number of LSTM units
Returns:
  Keras Model with encoder outputs and states
'''
def build_encoder(vocab_size, embedding_dim, units):
  inputs = layers.Input(shape=(None,)) # variable-length input sequence
  x = layers.Embedding(vocab_size, embedding_dim)(inputs)
  outputs, state_h, state_c = layers.LSTM(units, return_sequences=True, return_state=True)(x)
  return tf.keras.Model(inputs, [outputs, state_h, state_c])

In [48]:
# decoder with attention
'''
Builds an LSTM decoder with Luong-style attention

Parameters:
  vocab_size: size of output vocabulary
  embedding_dim: size of embedding vectors
  units: number of LSTM units
Returns:
  Keras Model producing predictions at each time step
'''
def build_decoder(vocab_size, embedding_dim, units):
  # decoder inputs
  inputs = layers.Input(shape=(None,)) # decoder input sequence
  encoder_outputs = layers.Input(shape=(None, units)) # encoder outputs
  state_h = layers.Input(shape=(units,)) # previous hidden state
  state_c = layers.Input(shape=(units,)) # previous cell state

  # embedding layer
  x = layers.Embedding(vocab_size, embedding_dim)(inputs)

  # LSTM step
  lstm_out, lstm_state_h, lstm_state_c = layers.LSTM(
      units,
      return_sequences=True,
      return_state=True
  )(x, initial_state=[state_h, state_c])

  # attention - compute context vector from encoder outputs
  attention = Attention()
  context, attention_weights = attention(lstm_out, encoder_outputs)

  # concatenate context with LSTM output
  concat = layers.Concatenate()([lstm_out, context])

  # final output projection to vocab
  outputs = layers.Dense(vocab_size, activation='softmax')(concat)

  return tf.keras.Model(
      [inputs, encoder_outputs, state_h, state_c],
      outputs
  )

The model uses a Seq2Seq framework with an LSTM encoder that processes the input sequence and outputs hidden states for each time step, along with the final hidden and cell states. The LSTM decoder generates the output sequence one token at a time, using its previous hidden state and input token at each step. Scaled dot-product (Luong) attention is applied in the decoder, where the decoder hidden state serves as the query, and the encoder outputs serve as keys and values. This produces a context vector at each decoding step, which is combined with the decoder state to compute the next token prediction. Attention is integrated entirely in the decoder, allowing it to selectively focus on different parts of the input sequence dynamically, improving handling of long sequences and enhancing prediction accuracy for each output token.

Part 3

Dataset: https://www.kaggle.com/datasets/alvations/tatoeba

Tatoeba is a crowd-sourced dataset made up of example sentences and their translations.

In [3]:
!unzip "archive (4)".zip

Archive:  archive (4).zip
  inflating: links.csv               
  inflating: sentences_detailed.csv  
  inflating: tatoeba-sentpairs.tsv   


In [4]:
import pandas as pd

# load TSV subset of Tatoeba sentence pairs (English-French)
def load_tatoeba(file_path, num_samples=10000):
  data = pd.read_csv(file_path, sep='\t')

  # filter only English-French pairs
  mask = (data['SRC LANG'] == 'eng') & (data['TRG LANG'] == 'fra')
  data = data[mask]

  # subset (num_samples) for small-scale training
  data = data[:num_samples]

  eng_sentences = data['SRC'].tolist()
  fra_sentences = data['TRG'].tolist()

  return eng_sentences, fra_sentences

In [5]:
# tokenization and padding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# tokenize and pad list of sentences
def tokenize(sentences):
  tokenizer = Tokenizer(filters='') # no filtering
  tokenizer.fit_on_texts(sentences)
  sequences = tokenizer.texts_to_sequences(sentences)
  padded = pad_sequences(sequences, padding='post') # pad at end
  return padded, tokenizer

In [6]:
# load sentences
eng_sentences, fra_sentences = load_tatoeba("tatoeba-sentpairs.tsv")

# add start and end tokens for decoder (teacher forcing)
fra_sentences = ["<start> " + s + " <end>" for s in fra_sentences]

# tokenize encoder (English) and decoder (French)
encoder_input, eng_tokenizer = tokenize(eng_sentences)
decoder_input, fra_tokenizer = tokenize(fra_sentences)

# prepare decoder target by shifting decoder input by 1
decoder_target = decoder_input[:, 1:]
decoder_input = decoder_input[:, :-1]

# train-test split
split = int(0.8 * len(encoder_input))

encoder_train = encoder_input[:split]
decoder_train = decoder_input[:split]
target_train = decoder_target[:split]

encoder_test = encoder_input[split:]
decoder_test = decoder_input[split:]
target_test = decoder_target[split:]

print("Number of training samples:", len(encoder_train))
print("Number of test samples:", len(encoder_test))
print("Sample English:", eng_sentences[:5])
print("Sample French:", fra_sentences[:5])

Number of training samples: 8000
Number of test samples: 2000
Sample English: ["Let's try something.", "Let's try something.", 'I have to go to sleep.', "Today is June 18th and it is Muiriel's birthday!", "Today is June 18th and it is Muiriel's birthday!"]
Sample French: ['<start> Essayons quelque chose\u202f! <end>', '<start> Tentons quelque chose\u202f! <end>', '<start> Je dois aller dormir. <end>', "<start> Aujourd'hui nous sommes le 18 juin et c'est l'anniversaire de Muiriel\u202f! <end>", "<start> Aujourd'hui c'est le 18 juin, et c'est l'anniversaire de Muiriel. <end>"]


Teacher forcing is applied during training so that at each decoding step, the decoder receives the actual next token from the target sequence rather than its own previous prediction. This guides the model to learn the correct sequence more quickly and reduces error propagation, especially in early training stages.

The dataset contains 8,000 training samples and 2,000 test samples. Each sample consists of an English sentence as the encoder input and a corresponding French sentence as the decoder target. The French sentences are framed with special tokens <start> at the beginning and <end> at the end to indicate where the decoder should begin and stop generating tokens. The sample lists show a few of these sentence pairs for reference.

In [53]:
# build encoder-decoder model
embedding_dim = 64
units = 128

encoder = build_encoder(len(eng_tokenizer.word_index)+1, embedding_dim, units)
decoder = build_decoder(len(fra_tokenizer.word_index)+1, embedding_dim, units)

# encoder outputs and states
enc_inputs = encoder.input
enc_outputs, state_h, state_c = encoder.output

# decder inputs and outputs
dec_inputs = decoder.input[0]
dec_outputs = decoder([dec_inputs, enc_outputs, state_h, state_c])

model = tf.keras.Model([enc_inputs, dec_inputs], dec_outputs)

In [54]:
# compile model with sparce categorical crossentropy
model.compile(
  optimizer='adam',
  loss='sparse_categorical_crossentropy',
  metrics=['accuracy']
)

# train model
model.fit(
  [encoder_train, decoder_train],
  target_train,
  batch_size=64,
  epochs=50
)

Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.8742 - loss: 1.7509
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.8946 - loss: 0.7948
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.8954 - loss: 0.7753
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.8963 - loss: 0.7541
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.8976 - loss: 0.7315
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.8990 - loss: 0.7108
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9004 - loss: 0.6914
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9016 - loss: 0.6733
Epoch 9/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9030 - loss: 0.6557
Epoch 10/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9039 - loss: 0.6375
Epoch 11/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9054 - loss: 0.6157
Epoch 12/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 

The training shows steady improvement in both loss and accuracy. The model starts with a loss of 1.7509 and accuracy of 0.8742 in the first epoch. By epoch 10, loss decreases to 0.6375 and accuracy rises to 0.9039. Midway through training, around epoch 25, the loss drops to 0.3811 and accuracy reaches 0.9248. By epoch 40, the model achieves a loss of 0.2034 and accuracy of 0.9558. Training continues improving gradually, and by the final epoch, epoch 50, the loss reaches 0.1328 while accuracy climbs to 0.9705. The decrease in loss and increase in accuracy are consistent and smooth across epochs, indicating the model is learning effectively and converging well. There are no abrupt jumps or oscillations in either metric.

In [72]:
# BLEU score
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

'''
Computes BLEU score between a reference sentence and a predicted sentence

Parameters:
  reference: ground truth sentence (string)
  prediction: model-generated sentence (string)

Returns:
  BLEU score (float) measuring similarity between reference and prediction
'''

def compute_bleu(reference, prediction):
  reference = [reference.split()] # list of reference words
  prediction = prediction.split() # list of predicted words
  smoothie = SmoothingFunction().method1
  return sentence_bleu(reference, prediction, smoothing_function=smoothie)

# evaluates trained model on a subset of test data using BLEU score
def evaluate_bleu():
  for idx in range(10):
      # predict next sequence
      pred = model.predict(
          [encoder_test[idx:idx+1], decoder_test[idx:idx+1]]
      )

      # take argmax for token indices
      pred_seq = pred.argmax(axis=-1)[0]

      # convert token indicies to words
      predicted_sentence = " ".join(
          [fra_tokenizer.index_word.get(token, '') for token in pred_seq]
      )

      reference = fra_sentences[split + idx]

      print("Pred:", predicted_sentence)
      print("Ref :", reference)
      print("BLEU:", compute_bleu(reference, predicted_sentence))

evaluate_bleu()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Pred: j'ai vis une ville ville pour loin. <end>                                                              
Ref : <start> <start> Je vis une petite maison au loin. <end> <end>
BLEU: 0.05709324736182451
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Pred: j'ai vis une jour au loin. <end>                                                               
Ref : <start> <start> Je vis un cottage au loin. <end> <end>
BLEU: 0.11443963624627547
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Pred: j'ai vis une jour pour loin. <end>                                                               
Ref : <start> <start> Je vis un chalet au loin. <end> <end>
BLEU: 0.05035983350735684
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Pred: nous avons vu la film, emmenée lac. <end>                                                              
Ref : <start> <start> Nous avons vu une montagne au loin. <end> <end>
BLEU: 0.0446778805361507
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Pred: tu pouvez le la 

The model uses sparse categorical cross-entropy as the loss function because the target sequences are integer-encoded. This loss measures how well the predicted probability distribution at each time step aligns with the actual next token. The model produces a probability distribution over the vocabulary at each decoding step, and argmax is used to select the most likely token as the predicted word.

The predicted sentences show frequent repetition of words, such as “ville ville” or “le le,” and include words that do not fit grammatically. The model captures some vague structure, like consistently including “au loin,” which results in BLEU scores between 0.02 and 0.11. These low scores reflect minimal overlap between predicted and reference sentences, with differences in key nouns or modifiers heavily penalizing BLEU. More epochs could be used for better convergence. Using teacher forcing during evaluation, feeding the full target sequence to the decoder, can cause repeated or nonsensical outputs. Tokenization issues could amplify repetition errors if the mapping from token indices to words includes unknowns or empty tokens. BLEU is sensitive to exact word matches, which lowers scores when the predicted sentence uses synonyms or slightly different phrasing. Proper autoregressive decoding predicts one token at a time, feeding each predicted token back into the decoder. Preprocessing steps like normalizing text and removing punctuation could help with the scores Regularization techniques like dropout or gradient clipping could also help stabilize training.

Part 4

In [9]:
# tokenize French
start_token = fra_tokenizer.word_index['<start>'] # integer index for <start>
end_token = fra_tokenizer.word_index['<end>'] # integer index for <end>

# rebuild split
split = int(0.8 * len(encoder_input))

encoder_train_full = encoder_input[:split]
decoder_train_full = decoder_input[:split]
target_train_full  = decoder_target[:split]

encoder_test = encoder_input[split:]
decoder_test = decoder_input[split:]
target_test  = decoder_target[split:]

# validation split
val_split = int(0.8 * len(encoder_train_full))

encoder_train = encoder_train_full[:val_split]
decoder_train = decoder_train_full[:val_split]
target_train  = target_train_full[:val_split]

encoder_val = encoder_train_full[val_split:]
decoder_val = decoder_train_full[val_split:]
target_val  = target_train_full[val_split:]
fra_val = fra_sentences[:split][val_split:]

In [10]:
# positional encoding
class PositionalEncoding(layers.Layer):
    '''
    Implements sinusoidal positional encoding for transformer models

    Parameters:
      d_model: dimensionality of the model embeddings
      max_len: maximum sequence length supported
    '''
    def __init__(self, d_model, max_len=1000):
      super().__init__()

      # compute sinusoidal positional encoding
      pos = np.arange(max_len)[:, np.newaxis]
      i = np.arange(d_model)[np.newaxis, :]
      angles = pos / np.power(10000, (2 * (i // 2)) / d_model)

      angles[:, 0::2] = np.sin(angles[:, 0::2]) # sin for even indicies
      angles[:, 1::2] = np.cos(angles[:, 1::2]) # cos for odd indicies

      self.pos_encoding = tf.cast(angles[np.newaxis, ...], tf.float32)

    '''
    Adds positional encoding to the input embeddings

    Parameters:
      x: input tensor of shape (batch_size, sequence_length, d_model)

    Returns:
      Tensor of same shape as input with positional encoding added
    '''
    def call(self, x):
      seq_len = tf.shape(x)[1]
      return x + self.pos_encoding[:, :seq_len, :] # add positional info to embeddings

In [11]:
# masks
'''
Creates a padding mask for sequences

Parameters:
  seq: input tensor of shape (batch_size, seq_len) - 0 indicates padding

Returns:
  mask tensor of shape (batch_size, 1, 1, seq_len) with 1s at padding positions
'''
def create_padding_mask(seq):
  mask = tf.cast(tf.math.equal(seq, 0), tf.float32)
  return mask[:, tf.newaxis, tf.newaxis, :] # (batch,1,1,seq_len)

'''
Creates a look-ahead mask for a sequence to prevent attention to future tokens

Parameters:
  size: length of the sequence

Returns:
  mask tensor of shape (size, size) with 1s in upper triangle (future positions) and 0s elsewhere
'''
def create_look_ahead_mask(size):
  return 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0) # upper triangular zeros mask future tokens

In [12]:
# scaled dot attention
'''
Computes scaled dot-product attention

Parameters:
  q: query tensor of shape (..., seq_len_q, depth)
  k: key tensor of shape (..., seq_len_k, depth)
  v: value tensor of shape (..., seq_len_v, depth_v)
  mask: (optional) mask tensor broadcastable to (..., seq_len_q, seq_len_k)

Returns:
  output: tensor of shape (..., seq_len_q, depth_v) after applying attention
  attention_weights: tensor of shape (..., seq_len_q, seq_len_k) representing attention distribution
'''
def scaled_dot_product_attention(q, k, v, mask=None):
  matmul_qk = tf.matmul(q, k, transpose_b=True)

  dk = tf.cast(tf.shape(k)[-1], tf.float32)
  scaled_attention_logits = matmul_qk / tf.math.sqrt(dk) # scale by sqrt(dk)

  if mask is not None:
      scaled_attention_logits += (mask * -1e9) # mask out padded or future tokens

  attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
  output = tf.matmul(attention_weights, v)

  return output, attention_weights

The scaled dot product attention function above is the same implementation from part 1, but rewritten using numpy.

In [13]:
# multi-headed attention
class MultiHeadAttention(layers.Layer):
    '''
    Implements multi-headed attention mechanism for transformer models

    Parameters:
      d_model: dimensionality of model embeddings
      num_heads: number of attention heads

    Returns:
      Keras Layer computing multi-head attention output
    '''
    def __init__(self, d_model, num_heads):
      super().__init__()
      self.num_heads = num_heads
      self.d_model = d_model
      self.depth = d_model // num_heads # split embedding dimension across heads

      self.wq = layers.Dense(d_model) # learnable projection for queries
      self.wk = layers.Dense(d_model) # learnable projection for keys
      self.wv = layers.Dense(d_model) # learnable projection for values
      self.dense = layers.Dense(d_model) # combine heads back

    '''
    Splits embedding into multiple heads

    Parameters:
      x: input tensor of shape (batch_size, seq_len, d_model)
      batch_size: integer batch size

    Returns:
      tensor of shape (batch_size, num_heads, seq_len, depth) ready for attention
    '''
    def split_heads(self, x, batch_size):
      # reshape for multiple attention heads
      x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
      return tf.transpose(x, perm=[0, 2, 1, 3]) # (batch, heads, seq_len, depth)

    '''
    Performs multi-head attention

    Parameters:
      q: query tensor of shape (batch_size, seq_len_q, d_model)
      k: key tensor of shape (batch_size, seq_len_k, d_model)
      v: value tensor of shape (batch_size, seq_len_v, d_model)
      mask: (optional) mask tensor broadcastable to (batch_size, num_heads, seq_len_q, seq_len_k)

    Returns:
      tensor of shape (batch_size, seq_len_q, d_model) after attention
    '''
    def call(self, q, k, v, mask):
      batch_size = tf.shape(q)[0]

      q = self.split_heads(self.wq(q), batch_size)
      k = self.split_heads(self.wk(k), batch_size)
      v = self.split_heads(self.wv(v), batch_size)

      scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v, mask)
      concat = tf.reshape(scaled_attention, (batch_size, -1, self.d_model)) # combine heads

      return self.dense(concat)

In [14]:
# encoder layer
class EncoderLayer(layers.Layer):
    '''
    Implements a single transformer encoder layer with multi-head attention and feedforward network

    Parameters:
      d_model: dimensionality of the model embeddings
      num_heads: number of attention heads
      dff: dimensionality of the feedforward network hidden layer

    Returns:
      Keras Layer computing one encoder layer output
    '''
    def __init__(self, d_model, num_heads, dff):
      super().__init__()
      self.mha = MultiHeadAttention(d_model, num_heads)
      self.ffn = tf.keras.Sequential([
          layers.Dense(dff, activation='relu'), # feedforward layer
          layers.Dense(d_model)
      ])

      self.norm1 = layers.LayerNormalization(epsilon=1e-6) # residual + layerNorm
      self.norm2 = layers.LayerNormalization(epsilon=1e-6)

    '''
    Performs forward pass of encoder layer

    Parameters:
      x: input tensor of shape (batch_size, seq_len, d_model)
      mask: (optional) mask tensor broadcastable to (batch_size, num_heads, seq_len, seq_len)

    Returns:
      tensor of shape (batch_size, seq_len, d_model) after attention and feedforward
    '''
    def call(self, x, mask):
      attn = self.mha(x, x, x, mask) # self-attention
      x = self.norm1(x + attn) # residual connection
      ffn = self.ffn(x) # feedforward
      return self.norm2(x + ffn)

In [15]:
# decoder layer
class DecoderLayer(layers.Layer):
    '''
    Implements a single transformer decoder layer with masked self-attention, cross-attention, and feedforward network

    Parameters:
      d_model: dimensionality of model embeddings
      num_heads: number of attention heads
      dff: dimensionality of feedforward network hidden layer

    Returns:
      Keras Layer computing one decoder layer output
    '''
    def __init__(self, d_model, num_heads, dff):
      super().__init__()
      self.mha1 = MultiHeadAttention(d_model, num_heads) # masked self-attention
      self.mha2 = MultiHeadAttention(d_model, num_heads) # cross-attention

      self.ffn = tf.keras.Sequential([
          layers.Dense(dff, activation='relu'), # first dense layer
          layers.Dense(d_model) # second dense layer
      ])

      # layer normalizations
      self.norm1 = layers.LayerNormalization(epsilon=1e-6)
      self.norm2 = layers.LayerNormalization(epsilon=1e-6)
      self.norm3 = layers.LayerNormalization(epsilon=1e-6)

    '''
    Performs forward pass of decoder layer

    Parameters:
      x: input tensor of shape (batch_size, target_seq_len, d_model)
      enc_output: encoder output tensor of shape (batch_size, input_seq_len, d_model)
      look_ahead_mask: mask to prevent attention to future positions (target sequence)
      padding_mask: mask to prevent attention to padding tokens in encoder output

    Returns:
      tensor of shape (batch_size, target_seq_len, d_model) after attention and feedforward
    '''
    def call(self, x, enc_output, look_ahead_mask, padding_mask):
      attn1 = self.mha1(x, x, x, look_ahead_mask) # masked self-attention
      x = self.norm1(x + attn1)

      attn2 = self.mha2(x, enc_output, enc_output, padding_mask) # cross-attention with encoder
      x = self.norm2(x + attn2)

      ffn = self.ffn(x)
      return self.norm3(x + ffn)

In [16]:
# transformer model
class Transformer(tf.keras.Model):
    '''
    Implements a basic transformer model for sequence-to-sequence tasks

    Parameters:
      eng_vocab_size: size of English input vocabulary
      fra_vocab_size: size of French output vocabulary

    Returns:
      Keras Model computing output token probabilities for a target sequence given an input sequence
    '''
    def __init__(self, eng_vocab_size, fra_vocab_size):
      super().__init__()
      self.d_model = 64 # embedding dimension

      # embedding layers for encoder and decoder
      self.enc_embedding = layers.Embedding(eng_vocab_size, self.d_model)
      self.dec_embedding = layers.Embedding(fra_vocab_size, self.d_model)

      # positional encoding layer
      self.pos_encoding = PositionalEncoding(self.d_model)

      # encoder and decoder layers
      self.enc_layers = [EncoderLayer(64, 2, 128) for _ in range(2)]
      self.dec_layers = [DecoderLayer(64, 2, 128) for _ in range(2)]

      # final output layer - generates probabilities over vocab
      self.final_layer = layers.Dense(fra_vocab_size, activation='softmax')

    '''
    Performs forward pass of the transformer model

    Parameters:
      inputs: tuple of (inp, tar)
          inp: input tensor of shape (batch_size, input_seq_len)
          tar: target tensor of shape (batch_size, target_seq_len)

    Returns:
      tensor of shape (batch_size, target_seq_len, fra_vocab_size) representing predicted token probabilities
    '''
    def call(self, inputs):
      inp, tar = inputs

      # create masks for padding and look-ahead
      enc_padding_mask = create_padding_mask(inp)
      dec_padding_mask = create_padding_mask(inp)
      look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
      dec_target_padding_mask = create_padding_mask(tar)

      # combined mask for decoder
      look_ahead_mask = create_look_ahead_mask(tf.shape(tar)[1])
      dec_target_padding_mask = create_padding_mask(tar)

      combined_mask = tf.maximum(
          dec_target_padding_mask,
          look_ahead_mask[tf.newaxis, tf.newaxis, :, :]
      )

      # encoder
      enc = self.enc_embedding(inp)
      enc *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
      enc = self.pos_encoding(enc)

      for layer in self.enc_layers:
          enc = layer(enc, enc_padding_mask)

      # decoder
      dec = self.dec_embedding(tar)
      dec *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
      dec = self.pos_encoding(dec)

      for layer in self.dec_layers:
          dec = layer(dec, enc, combined_mask, dec_padding_mask)

      # final output probabilities
      return self.final_layer(dec)

In [18]:
# loss function
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
  from_logits=False, reduction='none'
)

'''
Custom sparse categorical crossentropy loss function that ignores padding tokens

Parameters:
  y_true: ground truth tensor of shape (batch_size, seq_len)
  y_pred: predicted tensor of shape (batch_size, seq_len, vocab_size)

Returns:
  scalar loss averaged over non-padding tokens
'''
def loss_function(y_true, y_pred):
  # compute per-token loss
  loss = loss_object(y_true, y_pred)

  # ignore padding tokens in loss calculation
  mask = tf.cast(tf.not_equal(y_true, 0), dtype=loss.dtype)
  loss *= mask

  # average loss over non-padding tokens
  return tf.reduce_sum(loss) / tf.reduce_sum(mask)

# compile and train transformer
transformer = Transformer(
  eng_vocab_size=len(eng_tokenizer.word_index)+1,
  fra_vocab_size=len(fra_tokenizer.word_index)+1
)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

# compile with custom loss
transformer.compile(
  optimizer=optimizer,
  loss=loss_function,
  metrics=['accuracy']
)

# train model
transformer.fit(
  [encoder_train, decoder_train],
  target_train,
  validation_data=([encoder_val, decoder_val], target_val),
  batch_size=64,
  epochs=100
)

Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 50s 58ms/step - accuracy: 0.0144 - loss: 7.2903 - val_accuracy: 0.0160 - val_loss: 6.3664
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.0339 - loss: 5.2647 - val_accuracy: 0.0457 - val_loss: 4.7906
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0622 - loss: 3.7060 - val_accuracy: 0.0634 - val_loss: 3.7213
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0756 - loss: 2.7671 - val_accuracy: 0.0734 - val_loss: 3.1027
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0851 - loss: 2.1431 - val_accuracy: 0.0798 - val_loss: 2.7201
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.0935 - loss: 1.6597 - val_accuracy: 0.0847 - val_loss: 2.4422
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1011 - loss: 1.2741 - val_accuracy: 0.0886 - val_loss: 2.2482
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1079 - loss: 0.9627 -

The model starts with very high loss at 7.29 and very low accuracy of 1.4%. Validation loss begins at 6.37 with 1.6% accuracy. In the first few epochs, training loss drops rapidly, reaching 0.13 by epoch 13, while validation loss falls more slowly to about 1.83. Training accuracy goes to roughly 12% by epoch 13 and then plateaus, with validation accuracy reaching only 9.7–9.8% and staying almost constant. From epoch 14 onward, training loss continues decreasing toward near zero, while validation loss gradually rises from 1.83 to 2.07 by epoch 100. Training accuracy remains at 12.25%, showing that the model is memorizing the training data without generalizing. Validation accuracy stagnation and slowly increasing validation loss indicate severe overfitting. The model is likely predicting only the most frequent class, failing to learn meaningful patterns. The reasons for this could be class imbalance, overly high learning rate, label misalignment, not enough preprocessing, or insufficient model complexity. The epochs suggests that the model might require more regularization, adjusted learning rate, careful loss function selection, and class weighting or oversampling.

In [25]:
# autoregressive translation
'''
Generates an autoregressive translation from input sequence using trained Transformer

Parameters:
  input_seq: input tensor of shape (1, input_seq_len) representing tokenized source sentence
  max_len: maximum length of target sequence to generate (default 20)
  temperature: scaling factor for sampling randomness, lower values make predictions more deterministic (default 0.7)

Returns:
  Translated sentence as a string
'''
def translate(input_seq, max_len=20, temperature=0.7):
  output = [start_token] # initialize with <start> token

  for step in range(max_len):
      # pad current output sequence
      padded_output = tf.keras.preprocessing.sequence.pad_sequences(
          [output], maxlen=max_len, padding='post'
      )

      # get predictions from transformer
      preds = transformer.predict([input_seq, padded_output], verbose=0)
      probs = preds[0, len(output)-1, :] # predictions for last token

      # apply temperature scaling - controls how random sampling is
      probs = np.log(probs + 1e-8) / temperature
      probs = np.exp(probs)
      probs = probs / np.sum(probs) # normalize

      # prevent repeating last 2 tokens
      for token in set(output[-2:]):
          probs[token] = 0
      probs = probs / np.sum(probs) # re-normalize

      # sample next token
      next_token = np.argmax(probs)

      # stop if <end> token is generated
      if next_token == end_token:
          break

      # ignore padding/unknown tokens
      if next_token != 0:
          output.append(next_token)

  # convert token IDs to words
  words = [
      fra_tokenizer.index_word.get(i, '')
      for i in output[1:] # skip <start> token
      if i != 0
  ]

  return " ".join(words)

In [24]:
# evaluate using BLEU score
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

'''
Removes start and end tokens from a reference sentence

Parameters:
  sentence: string containing a tokenized sentence (may include <start> and <end> tokens)

Returns:
  cleaned sentence string without <start> and <end> tokens
'''
def clean_sentence(sentence):
  # remove start/end tokens from reference
  return " ".join([w for w in sentence.split() if w not in ['<start>', '<end>']])


'''
Computes BLEU score between a reference and a predicted sentence using NLTK

Parameters:
  ref: reference sentence string
  pred: predicted sentence string

Returns:
  BLEU score as a float
'''
def compute_bleu_tf(ref, pred):
  # split reference and prediction into tokens
  ref_tokens = ref.split()
  pred_tokens = pred.split()

  # if predictions empty bleu 0
  if len(pred_tokens) == 0:
      return 0

  # smoothing function used to avoid zero BLEU when n-grams are missing
  smoothie = SmoothingFunction().method1

  # compute BLEU score using NLTK
  return sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie)

# evaluates transformer model translations on first 10 validation samples
def evaluate():
  for i in range(10):
      pred = translate(encoder_val[i:i+1])
      ref = clean_sentence(fra_val[i])

      print("Pred:", pred)
      print("Ref :", ref)
      print("BLEU:", compute_bleu_tf(ref, pred))

evaluate()

Pred: pourrais devrait leur mot vrai dit semble soit votre leur
Ref : Nous lui laissâmes la décision finale.
BLEU: 0
Pred: pourrais devrait leur mot vrai dit semble soit votre leur
Ref : Nous lui avons laissé la décision finale.
BLEU: 0
Pred: distance. grande écoles sont à en retard. qui t'a je qu'un qu'il celui-ci ou venir 49% d'un que vu mort.
Ref : Nous pensons que nous avons traversé le pire.
BLEU: 0.009629943614188135
Pred: distance. grande écoles sont à la vérité. maison. assez peut-être que qu'il qu'un 49% et déjà 49% et que qui
Ref : Nous sommes à l'ère de l'énergie nucléaire.
BLEU: 0.009629943614188135
Pred: distance. grande écoles sont à la de répondre à tous que qu'il qu'un ou qu'il 49% d'un celui-ci que qui
Ref : Nous ne voulions pas, mais nous avons dû y aller.
BLEU: 0
Pred: facile. avons de que ce peux soit votre leur ai n'a qu'un été 49% d'un mort.
Ref : Nous nous tenions face à face.
BLEU: 0
Pred: distance. grande écoles sont à la vérité. maison. assez peut-être que qu'

The model’s predictions show extreme collapse and repetitiveness. Many outputs are nonsensical fragments or repeated token sequences, like “pourrais devrait leur mot vrai dit semble soit votre leur” or “distance. grande écoles sont à…”, which have almost no semantic or syntactic resemblance to the reference sentences. BLEU scores are nearly always 0, with rare predictions reaching around 0.01, reflecting almost no n-gram overlap. Even the slightly higher BLEU of 0.049 comes from a short sequence partially matching reference words but still failing to get meaningful content. Predictions usually include irrelevant numbers, pronouns, or disconnected words that do not correspond to the reference context.

I experimented with decoding strategies that introduce randomness. I applied temperature sampling to adjust the probability distribution over the vocabulary, and random sampling to avoid deterministic greedy outputs. These changes slightly increased variability in the outputs, producing fragments that occasionally resembled parts of the references. The outputs remained largely incoherent, repetitive, or structurally broken, and BLEU scores stayed extremely low.

The model memorizes token patterns without capturing sentence-level context or structure. Mode collapse, insufficient context learning, and limited generalization explain why even adding temperature and randomness could not produce fluent, semantically correct translations.

**Comparision Part 2 and 4 Models**

The Seq2Seq model was trained over 50 epochs with 125 steps per epoch. Training started with a relatively high loss of 1.7509 and an accuracy of 0.8742 in the first epoch. The model quickly improved by epoch 2, with the loss dropping to 0.7948 and accuracy rising to 0.8946. Over the next several epochs, the training loss steadily decreased while accuracy increased incrementally, demonstrating stable learning. By epoch 10, the model had reached an accuracy of 0.9039 and a loss of 0.6375. This upward trend continued, and by epoch 25, accuracy had reached 0.9248 with a loss of 0.3811. The model kept improving at a consistent pace, achieving an accuracy of 0.9705 and a loss of 0.1328 by the final epoch, suggesting the model had learned the training data effectively. Even with strong training metrics, BLEU score evaluation on the test sentences showed very low performance. Predicted translations often repeated words or were structurally different from the reference, leading to BLEU scores ranging roughly from 0.02 to 0.11, indicating that the model was overfitting the training set or struggling with generalization. Seq2Seq training steps were very fast (around 5 seconds per step), reflecting the simpler RNN/LSTM architecture and smaller computational overhead compared to a Transformer.

The simplified Transformer model was trained for 100 epochs with 100 steps per epoch. The model started with extremely low accuracy of 0.0144 and a very high loss of 7.2903, along with low validation accuracy (0.0160) and validation loss (6.3664). Over the first 10 epochs, there was fast improvement in training loss, dropping to 0.5050 by epoch 10, but accuracy remained low (0.1190). Even with the decreasing loss, validation metrics was still poor. From roughly epoch 12 onward, training loss continued decreasing, reaching extremely small values (around 0.001–0.0005), while validation accuracy stagnated around 0.097 and validation loss hovered near 1.9. BLEU scores computed on several sample predictions revealed extremely low performance. Most sentences had BLEU of 0, and the few nonzero scores were still small (0.0096, 0.0115, 0.0494), indicating that the model rarely produced outputs resembling the references. This confirms that despite near-zero training loss, the Transformer failed to generate coherent translations, likely due to overfitting or inability to generalize.

Several factors contribute to the performance differences between the models. The Seq2Seq model trained for 50 epochs and was evaluated directly on the test set using BLEU scores, which often reached meaningful values, whereas the Transformer, trained for 100 epochs, mostly produced near-zero BLEU scores even on the validation set. The Seq2Seq model likely benefited from a more structured decoder that aligns source and target sequences, allowing it to improve accuracy steadily and generate partially correct predictions. In contrast, the simplified Transformer could have had sufficient depth, training data, or tuning to generalize well, as reflected in its low validation accuracy and negligible BLEU scores. The Seq2Seq model demonstrated smoother and more interpretable loss reduction, while the Transformer’s loss decreased sharply to near-zero values without corresponding improvement in BLEU or validation accuracy, highlighting overfitting or poor optimization. The difference in runtime per epoch and step also shows the computational trade-offs. Seq2Seq is lightweight and fast per step, while Transformer requires significantly more resources initially. The evaluation methodology also differed. Seq2Seq used BLEU on the test set, while Transformer metrics were mostly validation loss/accuracy, making direct comparisons challenging. BLEU scores for the Seq2Seq were to be tested on test set and for the Transformer on validation set, which also explains differences in the scores. When first tested on test set, the BLEU scores for the Transformer were higher, but since the assignment asked for validation set, differences are seen in the comparision. The Seq2Seq model retained some ability to produce structured output, whereas the Transformer largely failed to generate coherent translations despite extremely low training loss.